<a href="https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/CIS_5450_Project_Difficulty_Topics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 5450 Project: Difficulty Topics
**Group Members:**
* **Shangyi Du**
* **Jingyi Gong**
* **Chenning Huang**

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


## Topic 1: Entity Linking
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=mH3thvhSHusP&line=1&uniqifier=1)

### Why we used this concept
We applied entity linking because our research aims to examine whether contextual factors beyond direct observations, such as environmental conditions, geographic context, and cultural or socioeconomic characteristics, may influence how UFO shapes are reported. The NUFORC sightings dataset alone lacks these additional predictors, so we incorporated multiple external datasets at the city and country levels. Since these contextual features are not directly aligned to sighting coordinates, entity linking was necessary to accurately assign the nearest valid environmental and human-contextual attributes to each UFO report. This enriched dataset provides a more comprehensive feature space that enables us to explore a wider range of potential influences in subsequent modeling and analysis.

### How we implemented it
We applied two entity linking:

1. **UFO Sightings → City Features (Nearest-Valid City Matching)**  
   In the city features dataset, `population`, `avg_annual_temp(°C)`, `temp_seasonality`, `annual_precipitation(mm)`, `elevation(m)`, and `viirs_annual_ave` contain missing values for some cities. If we simply merged each UFO sighting with the single nearest city, some UFO records would inherit missing feature values even though a slightly more distant nearby city has complete data.

   To address this, we implemented a nearest-non-null matching procedure:

   - For each UFO sighting, we identified the k nearest cities using BallTree K-nearest-neighbor search in projected coordinates.

   - For each feature, we examined these candidate cities in order of distance (nearest → farther).

   - We selected the closest city that actually has a valid (non-null) value for that feature.

   - We also recorded the distance to the city that supplied the data, producing a meaningful measure of spatial accuracy.


2. **UFO Sightings → Country Education Data (Reverse Geocoding)**  
   Because education indicators are reported at the country level, we:

   - Cleaned and numeric-converted the latitude/longitude fields.

   - Reverse-geocoded coordinates into standardized ISO-aligned country labels.

   - Merged the dataset via a normalized "country" key.


### Results & Interpretation
Entity linking allowed us to successfully augment each UFO sighting with environmental, geographic, and human-contextual features while avoiding missing-value propagation. By assigning the closest valid city for each contextual variable, this approach preserves geographic proximity and ensures that we do not introduce incomplete or unrealistic data. It also standardizes national attributes through reverse geocoding, associating each sighting with consistent country-level cultural and socioeconomic indicators. As a result, we obtained a more complete and robust feature set—one that maintains spatial realism, resolves missing data issues, and provides a reliable foundation for downstream modeling and analysis.

## Topic 2: Hyperparameter Tuning (XGBoost)
[Hyperlink](https://colab.research.google.com/drive/1Ulk5NQ70aEAgbZoy_I2DSzHznZfH2vI0?authuser=1#scrollTo=BuuChUUWejjY)
### Why we used this concept
XGBoost is a powerful model, but its performance depends heavily on hyperparameters such as depth, learning rate, number of estimators, and sampling ratios. Using default parameters led to underfitting and suboptimal accuracy. To build a stronger baseline and fair comparison, we used **Randomized Search via ParameterSampler**, which is more computationally efficient than an exhaustive GridSearch.

### How we implemented it
We created a parameter grid including:

- `max_depth`
- `learning_rate`
- `min_child_weight`
- `subsample`
- `colsample_bytree`
- `n_estimators`

Using `ParameterSampler`, we:

1. Randomly sampled combinations of parameters.  
2. Trained XGBoost on each sampled configuration.  
3. Measured performance using validation error.  
4. Selected the best‐performing set of hyperparameters.  
5. Used the tuned XGBoost model for both regression (ratings/popularity) and classification (performance tier).

### Results & Interpretation
Hyperparameter tuning consistently improved model performance:

- For **rating regression**, R² improved meaningfully relative to the untuned model.  
- For **popularity regression**, the tuned model captured nonlinear interactions far more effectively.  
- For **performance tier classification**, tuning improved accuracy and especially improved separation between High vs Low tiers.

This demonstrates a **methodical, justified improvement** over baseline models.



## Topic 3: Ensemble Models (VotingClassifier)
[Hyperlink](https://colab.research.google.com/drive/1Ulk5NQ70aEAgbZoy_I2DSzHznZfH2vI0?authuser=1#scrollTo=02eb354a)
### Why we used this concept
Different models capture different patterns in the data.  
- **Random Forest** handles nonlinear interactions and is robust to noise.  
- **XGBoost** captures complex boosted interactions, handles sparsity well, and typically yields stronger predictive performance.

Instead of selecting one “best” model, we implemented an **Ensemble VotingClassifier** to combine both models’ strengths. This aligns with our objective of improving performance tier prediction accuracy while maintaining stability and reducing overfitting.

Soft-voting ensembles often outperform individual models, especially in multi-class settings like our **High / Medium / Low performance tier** classification task.

---

### How we implemented it
We built a `VotingClassifier` using:

- A **RandomForestClassifier** (80–120 estimators, tuned depth and number of features)
- A **tuned XGBoost classifier** using the best hyperparameters discovered via Randomized Search (ParameterSampler)

We configured the ensemble as:

```python
VotingClassifier(
    estimators=[('rf', rf_clf), ('xgb', xgb_clf)],
    voting='soft',      # uses predicted probabilities
    n_jobs=4
)
